# EXP_060B — Alternative Combination 1: Swin-B + ViSoBERT + GMU + UncertaintyLoss
**Phase 6 | Promising Combination Validation**
Research question: Does social-media-matched text + hierarchical vision + gating synergize?
- Image: Swin-B | Text: ViSoBERT | Fusion: GMU | Loss: Uncertainty-Weighted | Seed: 42

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

Cloning into 'SE365'...
remote: Enumerating objects: 13415, done.
remote: Counting objects: 100% (286/286), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 13415 (delta 208), reused 188 (delta 113), pack-reused 13129 (from 1)
Receiving objects: 100% (13415/13415), 873.23 MiB | 19.96 MiB/s, done.
Resolving deltas: 100% (446/446), done.
/content/SE365


### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

total 1416
drwxr-xr-x  4 root root    4096 Jun 16 09:21 .
drwxr-xr-x 11 root root    4096 Jun 24 08:12 ..
drwxr-xr-x  2 root root 1437696 Jun 16 09:59 image
drwxr-xr-x  2 root root    4096 Jun 16 09:21 text


### STEP 4: Configure paths

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_060B_swinb_visobert_gmu_uncertainty'

BEST_IMAGE_EXP_ID = 'EXP_020B_swinb_xlmr_concat_mse'
BEST_TEXT_EXP_ID  = 'EXP_030D_bestimage_visobert_concat_mse'

DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts: {DRIVE_EXP_PATH}')

Artifacts: /content/drive/MyDrive/SE365/experiments/EXP_060B_swinb_visobert_gmu_uncertainty


### STEP 5: Load pretrained weights

In [ ]:
import os, shutil
os.makedirs('./checkpoints', exist_ok=True)
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_TEXT_EXP_ID}/best_model_train_text.pth', './checkpoints/best_model_train_text.pth')
print(f'Loaded text from {BEST_TEXT_EXP_ID}')
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_IMAGE_EXP_ID}/best_model_train_image.pth', './checkpoints/best_model_train_image.pth')
print(f'Loaded image from {BEST_IMAGE_EXP_ID}')

Loaded text from EXP_030D_bestimage_visobert_concat_mse
Loaded image from EXP_020B_swinb_xlmr_concat_mse


### STEP 6: Train

In [ ]:
!python main.py \
  --mode train_fusion \
  --fusion_type gmu \
  --text_model_name uitnlp/visobert \
  --image_model_name swin_base_patch4_window7_224 \
  --epochs 20 \
  --batch_size 16 \
  --lr 1e-5 \
  --grad_accum_steps 2 \
  --patience 5 \
  --loss_fn auto_weight \
  --unfreeze_text_layers 1 \
  --unfreeze_image_layers 1 \
  --seed 42 \
  --use_amp \
  --exp_id EXP_060B_swinb_visobert_gmu_uncertainty \
  --exp_dir ./experiments

====== MODE: TRAIN_FUSION ======
Using device: cuda
Seed: 42 | Experiment: EXP_060B_swinb_visobert_gmu_uncertainty
config.json: 100% 644/644 [00:00<00:00, 2.62MB/s]
sentencepiece.bpe.model: 100% 471k/471k [00:01<00:00, 239kB/s]
Loaded timm processor for swin_base_patch4_window7_224
pytorch_model.bin: 100% 390M/390M [00:05<00:00, 65.1MB/s]
Loading weights: 100% 197/197 [00:00<00:00, 25102.62it/s]
[transformers] XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly in

In [ ]:
!git pull

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.04 KiB | 534.00 KiB/s, done.
From https://github.com/lechihoang/SE365
   f0d2095..9430be8  main       -> origin/main
Updating f0d2095..9430be8
Fast-forward
 notebook/EXP_060B_swinb_visobert_gmu_uncertainty.ipynb   |  2 +-
 .../EXP_060C_efficientnetb3_phobert_film_huber.ipynb     |  2 +-
 test.py                                                  | 16 +++++++++++++++-
 3 files changed, 17 insertions(+), 3 deletions(-)


### STEP 7: Evaluate on Test Set
Evaluate the best model on the unseen test set to report final metrics and generate plots.

In [ ]:
!python test.py \
  --mode train_fusion \
  --fusion_type gmu \
  --text_model_name uitnlp/visobert \
  --image_model_name swin_base_patch4_window7_224 \
  --exp_id $EXP_ID \
  --exp_dir ./experiments \
  --save_path ./experiments/$EXP_ID



====== TESTING: TRAIN_FUSION ======
Device: cuda
Test samples: 600
Loading weights: 100% 197/197 [00:00<00:00, 29800.48it/s]
[transformers] XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loaded weights: ./experiments/EXP_060B_swinb_visobert_gmu_uncertainty/best_model_train_fusion.pth
/content/SE365/test.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is depre

### STEP 8: Save to Drive + print metrics


In [ ]:
import json
!cp -r ./experiments/$EXP_ID/* $DRIVE_EXP_PATH/

# --- VALIDATION METRICS ---
with open(f'./experiments/{EXP_ID}/metrics.json') as f:
    m = json.load(f)

print(f'\n=== {EXP_ID} Results (Validation) ===')
print(f"Loss (val)   : {m['loss']:.4f}")
print()
print("             MAE      RMSE      R2")
print(f"  food     : {m['mae_food']:.4f}   {m['rmse_food']:.4f}   {m['r2_food']:.4f}")
print(f"  price    : {m['mae_price']:.4f}   {m['rmse_price']:.4f}   {m['r2_price']:.4f}")
print(f"  atmos    : {m['mae_atmos']:.4f}   {m['rmse_atmos']:.4f}   {m['r2_atmos']:.4f}")
print(f"  service  : {m['mae_service']:.4f}   {m['rmse_service']:.4f}   {m['r2_service']:.4f}")
print(f"  overall  : {m['mae_overall']:.4f}   {m['rmse_overall']:.4f}   {m['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {m['mean_mae']:.4f}")
print(f"  aspect_mae : {m['aspect_mae']:.4f}")
print(f"  overall_mae: {m['overall_mae']:.4f}")

# --- TEST METRICS ---
with open(f'./experiments/{EXP_ID}/test_metrics.json') as f:
    t = json.load(f)

print(f'\n=== {EXP_ID} Results (Test) ===')
print()
print("             MAE      RMSE      R2")
print(f"  food     : {t['mae_food']:.4f}   {t['rmse_food']:.4f}   {t['r2_food']:.4f}")
print(f"  price    : {t['mae_price']:.4f}   {t['rmse_price']:.4f}   {t['r2_price']:.4f}")
print(f"  atmos    : {t['mae_atmos']:.4f}   {t['rmse_atmos']:.4f}   {t['r2_atmos']:.4f}")
print(f"  service  : {t['mae_service']:.4f}   {t['rmse_service']:.4f}   {t['r2_service']:.4f}")
print(f"  overall  : {t['mae_overall']:.4f}   {t['rmse_overall']:.4f}   {t['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {t['mean_mae']:.4f}")
print(f"  aspect_mae : {t['aspect_mae']:.4f}")
print(f"  overall_mae: {t['overall_mae']:.4f}")



=== EXP_060B_swinb_visobert_gmu_uncertainty Results (Validation) ===
Loss (val)   : 2.8061

             MAE      RMSE      R2
  food     : 1.2569   1.7459   0.4209
  price    : 1.2702   1.7682   0.3000
  atmos    : 1.2446   1.6361   0.3103
  service  : 1.2922   1.7320   0.4149
  overall  : 1.0864   1.4817   0.4608

  mean_mae   : 1.2300
  aspect_mae : 1.2660
  overall_mae: 1.0864

=== EXP_060B_swinb_visobert_gmu_uncertainty Results (Test) ===

             MAE      RMSE      R2
  food     : 1.2117   1.6469   0.5091
  price    : 1.1964   1.5779   0.4068
  atmos    : 1.2522   1.6140   0.3117
  service  : 1.1949   1.6348   0.4366
  overall  : 1.0185   1.3408   0.5432

  mean_mae   : 1.1747
  aspect_mae : 1.2138
  overall_mae: 1.0185
